# 11 – LlamaIndex PDF RAG with Node Parsers

**Learning Goals:**
- Use LlamaIndex for document ingestion and indexing
- Compare node parsing strategies (recursive, fixed)
- Integrate with ChromaDB for persistence
- Query using LlamaIndex QueryEngine

**What we'll build:**
1. Load PDFs using LlamaIndex SimpleDirectoryReader
2. Test node parser strategies (SentenceSplitter, TokenTextSplitter)
3. Embed and store in ChromaDB
4. Query with QueryEngine

**Persistence:**
- `./artifacts/chroma/llamaindex_recursive/`
- `./artifacts/chroma/llamaindex_fixed/`
- `./artifacts/manifests/llamaindex_{strategy}.json`


In [ ]:
# ⚙️ Global Config & Services (using centralized modules)

import json
import sys
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# Add parent directory to path and change to project root
import os

# Get the current directory and navigate to project root
current_dir = Path.cwd()
if current_dir.name == "homework":
    project_root = current_dir.parent
elif current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

# Change to project root and add to path
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"📂 Working directory: {os.getcwd()}")

from src.services.llm_services import (
    load_config,
    get_llamaindex_llm,
    get_llamaindex_embeddings,
    validate_api_keys,
    print_config_summary
)

# Load environment variables
load_dotenv()

# Load configuration from config.yaml (now we're in project root)
config = load_config("src/config/config.yaml")

# Validate API keys
validate_api_keys(config, verbose=True)

# Print summary
print_config_summary(config)


In [ ]:
# Configure LlamaIndex Settings globally using factories from llm_services
from llama_index.core import Settings

Settings.llm = get_llamaindex_llm(config)
Settings.embed_model = get_llamaindex_embeddings(config)
Settings.chunk_size = config.get("chunk_size", 800)
Settings.chunk_overlap = config.get("chunk_overlap", 150)

print(f"✅ LlamaIndex configured")
print(f"  LLM: {config['llm_provider']} / {config['llm_model']}")
print(f"  Embeddings: {config['text_emb_provider']} / {config['text_emb_model']}")

# Verify API key with test completion
print("\n🔍 Testing LLM API connection...")
try:
    test_response = Settings.llm.complete("Say 'API working!' if you can read this.")
    test_msg = str(test_response)[:50]
    print(f"✅ LLM API verified: {test_msg}")
except Exception as e:
    print(f"❌ LLM API test failed: {e}")
    print("⚠️  Please check your .env file and API key configuration.")


---

## Step 1: Load Documents

Using LlamaIndex's `SimpleDirectoryReader` to load PDFs.


In [ ]:
from llama_index.core import SimpleDirectoryReader, Document as LlamaDocument

pdf_dir = Path(config["data_root"]) / "pdfs"
pdf_dir.mkdir(parents=True, exist_ok=True)

# Try loading from directory
try:
    reader = SimpleDirectoryReader(str(pdf_dir), recursive=False)
    documents = reader.load_data()
except Exception as e:
    print(f"⚠️  Could not load from {pdf_dir}: {e}")
    
    # Create sample document
    sample_content = """# Common Skin Diseases and Conditions

Understanding skin diseases is essential for proper care and treatment. The skin is the largest organ and serves as a protective barrier against pathogens and environmental hazards.

## Inflammatory and Autoimmune Disorders

### Eczema (Atopic Dermatitis)
Eczema is a chronic inflammatory condition marked by itchy, dry, and red skin. It affects 10-20% of children and often has a genetic component involving skin barrier dysfunction. Treatment includes daily moisturizing, avoiding triggers, and topical anti-inflammatory medications.

### Psoriasis
Psoriasis is an autoimmune condition causing thick, silvery scales and red plaques due to rapid skin cell turnover. Treatment options include topical corticosteroids, phototherapy, and systemic medications.

### Vitiligo
Vitiligo causes loss of skin pigment due to destruction of melanocytes, resulting in white patches. Treatment requires strict sun protection and may include topical medications or phototherapy.

## Infectious Skin Diseases

### Fungal Infections
Ringworm (tinea) causes circular, red, scaly patches. Common types include athlete's foot and jock itch. Treatment involves topical or oral antifungal medications like terbinafine.

### Bacterial Infections
Impetigo causes honey-colored crusts and requires antibiotics. Cellulitis is a deeper infection causing swelling and redness that needs prompt medical attention.

## General Skin Care Principles

Daily moisturizing helps maintain the skin barrier and reduce dryness. Broad-spectrum SPF 30+ sunscreen protects against UV damage and prevents skin cancer. Identify and avoid personal triggers including irritants, allergens, and stress."""
    
    documents = [LlamaDocument(text=sample_content, metadata={"source": "skin_diseases_intro.txt"})]

print(f"✅ Loaded {len(documents)} documents")
print(f"  Total characters: {sum(len(d.text) for d in documents):,}")


---

## Step 2: Node Parsing Strategies

LlamaIndex uses node parsers to chunk documents. We'll test:
1. **SentenceSplitter** (recursive-like, respects sentences)
2. **SimpleNodeParser** with fixed chunking


In [ ]:
from llama_index.core.node_parser import SentenceSplitter, SimpleNodeParser

def get_node_parser(strategy: str, chunk_size: int = 800, chunk_overlap: int = 150):
    """
    Return a LlamaIndex node parser based on strategy.
    
    Args:
        strategy: One of "recursive" or "fixed"
        chunk_size: Target size in characters/tokens
        chunk_overlap: Overlap between consecutive nodes
        
    Returns:
        A LlamaIndex node parser instance
    """
    ### START CODE HERE ### (≈ 15-20 lines)
    # YOUR CODE HERE
    # HINTS:
    # LlamaIndex uses "node parsers" (similar to LangChain's text splitters)
    #
    # 1. if strategy == "recursive":
    #    Return SentenceSplitter with:
    #    - chunk_size=chunk_size          # Target size (flexible at sentence boundaries)
    #    - chunk_overlap=chunk_overlap    # Overlap between nodes
    #    SentenceSplitter respects sentence boundaries (like RecursiveCharacterTextSplitter)
    #
    # 2. elif strategy == "fixed":
    #    Return SimpleNodeParser.from_defaults() with:
    #    - chunk_size=chunk_size          # Fixed size
    #    - chunk_overlap=chunk_overlap    # Overlap
    #    from_defaults() is a factory method
    #
    # 3. else:
    #    Raise ValueError(f"Unknown strategy: {strategy}")
    #
    # NOTE: LlamaIndex nodes are like LangChain Documents
    # They have .text (content) and .metadata (dict)
    
    raise NotImplementedError("Complete the get_node_parser function")
    ### END CODE HERE ###


# Parse documents with both strategies
strategies = ["recursive", "fixed"]
node_results = {}

for strategy in strategies:
    parser = get_node_parser(strategy)
    nodes = parser.get_nodes_from_documents(documents)  # get_nodes_from_documents: Parse into nodes
    
    # Add strategy to metadata for tracking
    for node in nodes:
        node.metadata["parser"] = strategy
    
    node_results[strategy] = nodes
    print(f"✅ {strategy:10s}: {len(nodes):4d} nodes")

print(f"\nExample node (recursive):")
print(f"  Length: {len(node_results['recursive'][0].text)} chars")
print(f"  Text: {node_results['recursive'][0].text[:150]}...")
print(f"  Metadata: {node_results['recursive'][0].metadata}")


---

## Step 3: Build ChromaDB Indices

For each strategy, create a ChromaDB vector store and index.


In [ ]:
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

chroma_root = Path(config["artifacts_root"]) / "chroma"
chroma_root.mkdir(parents=True, exist_ok=True)

indices = {}

for strategy in strategies:
    collection_name = f"llamaindex_{strategy}"
    persist_dir = str(chroma_root / collection_name)
    
    print(f"Building index: {collection_name}...")
    
    ### START CODE HERE ### (≈ 15-18 lines)
    # YOUR CODE HERE
    # HINTS:
    # LlamaIndex integrates with ChromaDB differently than LangChain
    #
    # 1. Create a ChromaDB persistent client:
    #    chroma_client = chromadb.PersistentClient(path=persist_dir)
    #    - PersistentClient: Saves to disk (survives restarts)
    #    - path: Directory where ChromaDB stores data
    #
    # 2. Get or create a collection:
    #    chroma_collection = chroma_client.get_or_create_collection(collection_name)
    #    - get_or_create: Loads existing or creates new
    #    - collection_name: Unique identifier for this collection
    #
    # 3. Wrap in LlamaIndex ChromaVectorStore:
    #    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    #    - Adapter that lets LlamaIndex use ChromaDB
    #
    # 4. Create storage context:
    #    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    #    - from_defaults: Factory method
    #    - Tells LlamaIndex where to persist embeddings
    #
    # 5. Build VectorStoreIndex:
    #    index = VectorStoreIndex(
    #        nodes=node_results[strategy],      # Nodes to embed and index
    #        storage_context=storage_context,   # Where to store
    #    )
    #    - Automatically embeds all nodes using Settings.embed_model
    #    - Stores vectors in ChromaDB
    #
    # 6. Store the index: indices[strategy] = index
    
    raise NotImplementedError("Complete the LlamaIndex ChromaDB index creation")
    ### END CODE HERE ###
    
    print(f"  ✅ Persisted to {persist_dir}")
    print(f"  ✅ {len(node_results[strategy])} nodes indexed")

print(f"\n✅ All indices built!")


### Save Manifests


In [ ]:
manifests_dir = Path(config["artifacts_root"]) / "manifests"
manifests_dir.mkdir(parents=True, exist_ok=True)

for strategy in strategies:
    manifest = {
        "collection_name": f"llamaindex_{strategy}",
        "framework": "llamaindex",
        "strategy": strategy,
        "embedding_model": config["text_emb_model"],
        "embedding_provider": config["text_emb_provider"],
        "normalize": config["normalize_embeddings"],
        "chunk_size": 800,
        "chunk_overlap": 150,
        "num_nodes": len(node_results[strategy]),
        "num_documents": len(documents),
        "created_at": datetime.now().isoformat(),
    }
    
    manifest_path = manifests_dir / f"llamaindex_{strategy}.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    
    print(f"✅ Manifest saved: {manifest_path.name}")


---

## Step 4: Query with QueryEngine

LlamaIndex's QueryEngine handles retrieval + synthesis.


In [ ]:
from llama_index.core.prompts import PromptTemplate

# Custom RAG prompt
qa_prompt_tmpl = """You are a concise assistant. Use only the provided context to answer the question.
If the information is insufficient, say "I don't know." Keep answers under 6 sentences.

Context:
{context_str}

Question: {query_str}

Answer:"""

QA_PROMPT = PromptTemplate(qa_prompt_tmpl)

# Build query engines
### START CODE HERE ### (≈ 10-12 lines)
# YOUR CODE HERE
# HINTS:
# LlamaIndex QueryEngine is similar to LangChain's RetrievalQA
#
# 1. Create empty dictionary: query_engines = {}
#
# 2. Loop through strategies:
#    for strategy in strategies:
#
# 3. Create query engine from index:
#    engine = indices[strategy].as_query_engine(
#        similarity_top_k=3,           # similarity_top_k: Number of nodes to retrieve
#                                      # Like "k" in LangChain retrievers
#                                      # Typical values: 3-10
#        
#        text_qa_template=QA_PROMPT,   # text_qa_template: Custom prompt template
#                                      # Similar to LangChain's chain_type_kwargs
#                                      # Optional: response_mode parameter
#                                      # Options: "compact", "tree_summarize", "simple_summarize"
#    )
#
# 4. Store engine: query_engines[strategy] = engine
#
# 5. After loop, print: "✅ Query engines ready for all strategies"

raise NotImplementedError("Complete the query engine creation")
### END CODE HERE ###


---

## Interactive Demo

Query the indices and see node previews + answers.


In [ ]:
test_queries = [
    "What causes eczema and atopic dermatitis?",
    "How do you treat fungal infections like ringworm?",
    "What are the recommended treatments for psoriasis?",
]

selected_strategy = "recursive"  # Change to "fixed" to compare

print(f"📊 Testing strategy: {selected_strategy}\n")
print("=" * 80)

for query in test_queries:
    print(f"\n❓ Query: {query}\n")
    
    engine = query_engines[selected_strategy]
    response = engine.query(query)
    
    print("📚 Retrieved nodes:")
    for i, node in enumerate(response.source_nodes, 1):
        source = node.metadata.get("source", "unknown")
        parser = node.metadata.get("parser", "?")
        score = node.score if hasattr(node, 'score') else 0.0
        print(f"  [{i}] {source} (parser: {parser}, score: {score:.3f})")
        print(f"      {node.text[:120]}...\n")
    
    print(f"💡 Answer:\n{response.response}\n")
    print("=" * 80)


---

## Summary

**What we learned:**
- ✅ LlamaIndex provides unified abstractions for RAG
- ✅ Node parsers chunk documents while preserving structure
- ✅ VectorStoreIndex + ChromaDB = persistent retrieval
- ✅ QueryEngine handles retrieval + synthesis

**LlamaIndex vs LangChain:**
- LlamaIndex: More opinionated, RAG-first design
- LangChain: More flexible, composable chains
- Both integrate well with ChromaDB

**Artifacts:**
- `./artifacts/chroma/llamaindex_recursive/`
- `./artifacts/chroma/llamaindex_fixed/`
- `./artifacts/manifests/llamaindex_{strategy}.json`
